# Guide to Train Machine Learning Models on tracebloc 🚀

This notebook walks you through training an ML model on the tracebloc platform — from connecting your account to launching a training run.

**What you'll do:**
1. Connect to your tracebloc account
2. Upload a model & weights
3. Link the model with a dataset
4. Configure a training plan
5. Start training

This guide takes about **10–15 minutes** to complete.

## Prerequisites

Before you begin, make sure you have:

- ✅ A **tracebloc account** — [Sign up here](https://ai.tracebloc.io/signup) if you don't have one
- ✅ **Joined a use case** — you need an active use case with a dataset. [How to join a use case →](https://docs.tracebloc.io/join-use-case/explore-use-case)
- ✅ A **model file** (`.py`) compatible with the dataset — you can use one from the [tracebloc model zoo](https://github.com/tracebloc/model-zoo) or bring your own. [Model structure requirements →](https://docs.tracebloc.io/join-use-case/model-optimization)

📖 **Full documentation:** [docs.tracebloc.io](https://docs.tracebloc.io)

💡 **Prefer Google Colab?** [Open this guide in Colab](https://colab.research.google.com/github/tracebloc/start-training/blob/main/notebooks/traceblocTrainingGuide.ipynb) — runs entirely in your browser, no local setup needed. Once it opens, do **File → Save a copy in Drive** so your edits persist.

🐍 **Python** — the SDK declares the versions it supports in its own package metadata, and that is the only authority; this notebook does not restate it. The install cell below runs pip and, if the install fails, prints the exact range pip read together with the version you are on. A current default `python3` on macOS is often *ahead* of the supported range, so if you are running locally that is the first thing to check.

---
## 1. Connect to tracebloc

First, install the tracebloc package and log in with your tracebloc account email and password.

In [ ]:
# Install the tracebloc SDK, then explain whatever happened.
#
# This runs pip and reads pip's own answer. It deliberately does NOT check your
# Python version against a range written here first: that range would be a
# second copy of the SDK's `requires-python`, in a different repo, with nothing
# keeping the two in step. The first version of this cell did exactly that,
# went stale, and printed "OK" to a 3.14 user whose install then failed --
# a false all-clear (backend#2862, review on start-training#83).
#
# pip already knows the real bound and prints it, even under `-q`. That is the
# only copy that cannot be wrong.
import re
import subprocess
import sys

# Other extras: [sklearn] [catboost] [lightgbm] [xgboost] [lifelines]
# [scikit-survival] [all]. TensorFlow uploads were removed in SDK 1.0.0, so
# there is no [tensorflow] extra.
SPEC = "tracebloc[pytorch]>=0.14.0"

running = f"{sys.version_info[0]}.{sys.version_info[1]}"
print(f"Python {sys.version.split()[0]}")

proc = subprocess.run(
    [sys.executable, "-m", "pip", "install", SPEC],
    capture_output=True,
    text=True,
    check=False,  # the whole point of this cell is to handle a failure itself
)
output = proc.stdout + proc.stderr

if proc.returncode == 0:
    # Success side: prove it actually imports. A partial install otherwise
    # looks like success right up to the `from tracebloc import User` below.
    try:
        from importlib.metadata import version

        import tracebloc  # noqa: F401

        print(f"OK - tracebloc {version('tracebloc')} installed and imports.")
    # Broad on purpose: a half-installed dependency can raise almost anything
    # (ImportError, OSError from a bad .so, ValueError from a truncated wheel),
    # and a diagnostic cell must not become the crash it is reporting.
    except Exception as exc:  # noqa: BLE001
        print(f"pip succeeded but importing tracebloc did not: {exc!r}")
        print("Restart the runtime (Runtime -> Restart session) and re-run this cell.")
else:
    print(output[-1500:])
    # pip names the true bound for every candidate it skipped, in one line:
    #   "Ignored the following versions that require a different python
    #    version: 1.0.2 Requires-Python >=3.11,<3.13; 1.0.3 Requires-Python
    #    >=3.11,<3.13"
    # Entries are separated by "; " and each specifier CONTAINS commas, so the
    # character class must exclude ";" and newline but NOT ",". Excluding ","
    # captured ">=3.11" out of ">=3.11,<3.13" and dropped the upper bound --
    # i.e. it hid the too-new-Python case this cell exists to explain
    # (Bugbot, start-training#83). Listed oldest-first, so the last match is
    # the newest release.
    bounds = re.findall(r"Requires-Python\s+([^\n;]+)", output)
    if bounds:
        print(
            f"\nWhat this means: you are on Python {running}, and the newest "
            f"release on the index supports {bounds[-1].strip()}.\n"
            "The package is not missing -- your interpreter is outside the range "
            "it declares, and no version pin changes that.\n"
            "  - On Colab the runtime's Python is not selectable. Please report "
            "the version printed above at\n"
            "    https://github.com/tracebloc/start-training/issues\n"
            "  - Locally, make a virtualenv on an interpreter inside that range "
            "and register it as this\n"
            "    notebook's kernel (`<python-inside-the-range> -m venv .venv`)."
        )
    else:
        # No `Requires-Python` in the output does NOT mean the interpreter is
        # fine: pip only started printing that line in newer versions. Measured
        # on pip 21.2.4 (the macOS system Python 3.9 default), a genuine
        # version mismatch prints just "from versions: ..." with no range at
        # all -- so claiming "not a version problem" here would be wrong for
        # exactly the local user this cell is meant to help.
        print(
            f"\nThe install failed and pip printed no supported-Python range, "
            f"so this cell cannot tell you whether Python {running} is the "
            "cause. Two things to rule out, in order:\n"
            "  - An older pip does not print that range at all. Run\n"
            f"    `{sys.executable} -m pip install --upgrade pip`, then re-run "
            "this cell -- if it was a version\n"
            "    mismatch, the upgraded pip will say so.\n"
            "  - Otherwise it is usually the index rather than the package: an "
            "unreachable index, a custom\n"
            "    --index-url, or private-index authentication.\n"
            "The pip output above is the authority."
        )

In [ ]:
from tracebloc import User

# This will prompt you for your tracebloc email and password
user = User()

**Expected output:** You'll see a prompt asking for your email and password. After entering them, you should see a confirmation that you're logged in.

⚠️ **If login fails:**
- Double-check your email and password at [ai.tracebloc.io](https://ai.tracebloc.io)
- Make sure you've verified your email address
- If you don't have an account yet, [sign up here](https://ai.tracebloc.io/signup)

---
## 2. Upload model & weights file

Next, upload your model file to the platform. You have two options:

### Option A: Use a model from the tracebloc model zoo

The [tracebloc model zoo](https://github.com/tracebloc/model-zoo) has ready-to-use models for common tasks:

| Task | Framework | Path |
|------|-----------|------|
| Image classification | PyTorch / TensorFlow | `model_zoo/image_classification/` |
| Object detection | PyTorch | `model_zoo/object_detection/pytorch/` |
| Text classification | PyTorch | `model_zoo/text_classification/pytorch/` |
| Tabular classification | PyTorch / Sklearn | `model_zoo/tabular_classification/` |
| Tabular regression | PyTorch / Sklearn | `model_zoo/tabular_regression/` |
| Time series forecasting | PyTorch | `model_zoo/time_series_forecasting/pytorch/` |
| Semantic segmentation | PyTorch | `model_zoo/semantic_segmentation/pytorch/` |
| Keypoint detection | PyTorch | `model_zoo/keypoint_detection/pytorch/` |
| Time-to-event prediction | PyTorch / Lifelines / Scikit-survival | `model_zoo/time_to_event_prediction/` |

Clone the model zoo and pick a model that fits your use case:

In [ ]:
# Clone the tracebloc model zoo (skipped if it's already present)
![ -d ../model-zoo ] && echo "model-zoo already present - skipping clone" || git clone https://github.com/tracebloc/model-zoo.git ../model-zoo

In [ ]:
# List available models
!ls ../model-zoo/model_zoo/

### Option B: Use your own model

You can upload your own model file. Place your `.py` file in the working directory or provide the full path.

Make sure your model follows the [model structure requirements](https://docs.tracebloc.io/join-use-case/model-optimization).

In [ ]:
# Upload your model file to tracebloc
# Replace the path with your actual model file location
MODEL_PATH = "../model-zoo/model_zoo/image_classification/pytorch/densenet.py"  # <-- change this

user.upload_model(MODEL_PATH)

**Expected output:** A confirmation message showing the model was uploaded successfully.

💡 **Loading weights?** Follow this naming convention:
- Model file: `mymodel.py`
- Weights file: `mymodel_weights.pkl`

The weights file must be in the same directory as the model file.

```python
# To upload with pretrained weights:
user.upload_model(MODEL_PATH, weights=True)
```

---
## 3. Link uploaded model with dataset

Now connect your uploaded model to a dataset from your use case.

**Where to find the Dataset ID:**
1. Go to [ai.tracebloc.io](https://ai.tracebloc.io) and open your use case
2. Copy the ID shown next to **Dataset** in the use case panel
3. Paste it below

The dataset ID is a short alphanumeric string (e.g., `DKbtefZy`).

In [ ]:
# Paste your Dataset ID here
DATASET_ID = "YOUR_DATASET_ID_HERE"  # <-- replace with your actual dataset ID (e.g., "DKbtefZy")

training = user.link_model_dataset(DATASET_ID)

**Expected output:** A confirmation that the model and dataset are linked.

⚠️ **If this fails:**
- Make sure the dataset ID is correct (check your use case panel)
- Your model must be compatible with the dataset (e.g., an image classification model for an image dataset)

---
## 4. Set training plan

Configure your training parameters. Start by naming your experiment, then adjust any parameters you need.

| Command | Description | Example |
|---------|-------------|--------|
| `training.experiment_name("...")` | Name your experiment | `training.experiment_name("My first run")` |
| `training.epochs(n)` | Number of training epochs | `training.epochs(10)` |
| `training.optimizer("...")` | Set optimizer | `training.optimizer("adam")` |
| `training.learning_rate({...})` | Set learning rate | `training.learning_rate({"type": "constant", "value": 0.001})` |
| `training.validation_split(n)` | Validation split | `training.validation_split(0.2)` |
| `training.get_training_plan()` | View the full training plan | |
| `training.reset_training_plan()` | Reset to defaults | |

For all available parameters, see the [Hyperparameters reference](https://docs.tracebloc.io/join-use-case/hyperparameters).

In [ ]:
# Set experiment name
training.experiment_name("My Experiment")

# Set training parameters
training.epochs(10)

# Review your training plan
training.get_training_plan()

**Expected output:** A summary showing all training parameters including:
- **Training Description** — experiment name, model name, objective
- **Dataset Parameters** — dataset ID, size, classes
- **Training Parameters** — epochs, cycles, batch size, validation split
- **Hyperparameters** — optimizer, loss function, learning rate, callbacks
- **Augmentation Parameters** — data augmentation settings

Review these carefully before starting. Adjust any values using the commands in the table above.

---
## 5. Start training

Everything configured? Launch the training run:

In [ ]:
training.start()  # start the experiment as configured above

## What happens next?

Your model is now being trained on the tracebloc infrastructure. Here's what to expect:

1. **Training starts** — the model will begin training on the linked dataset inside a secure environment
2. **Monitor progress** — go to your use case on [ai.tracebloc.io](https://ai.tracebloc.io) to see training status and logs
3. **View results** — once training completes, check the leaderboard in your use case to see how your model performed
4. **Compare models** — if other team members or vendors have submitted models, you can compare performance metrics side by side

Training time depends on your dataset size, model complexity, and the number of epochs. A typical training run takes a few minutes to a few hours.

📖 **Learn more:** [How to evaluate models →](https://docs.tracebloc.io/join-use-case/model-evaluation)

---
## Logout

When you're done, log out to end your session:

In [ ]:
user.logout()

---
## Need help?

- 📖 [Documentation](https://docs.tracebloc.io)
- 📧 [support@tracebloc.io](mailto:support@tracebloc.io)
- 🐛 [Open an issue](https://github.com/tracebloc/start-training/issues)
- 💬 [Discord](https://discord.gg/tracebloc)